In [1]:
import pandas as pd
from helpers import get_factor, get_price

In [2]:
CDF = pd.read_csv("../production/CDF.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

/tmp/ipykernel_1036702/3001582791.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))


In [3]:
smard = pd.read_csv("Gro_handelspreise_202401010000_202501010000_Stunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")

/tmp/ipykernel_1036702/2825335293.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
/tmp/ipykernel_1036702/2825335293.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")


In [4]:
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [5]:
CDF2 = CDF.loc[CDF.produced_at > "2023-12-31 23:50"].loc[CDF.produced_at < "2025-01-01 00:00"]

In [6]:
dataset = CDF2.merge(seem, left_on="variable", right_on="sseid")

In [7]:
magic = dataset.groupby(["produced_at", "plantid"]).sum()

In [8]:
magic2 = magic[["value"]]

In [9]:
#magic2.sort_values(["produced_at", "value"])

In [10]:
production = magic2.reset_index()
production["produced_at"] = pd.to_datetime(production["produced_at"], format="mixed")

In [11]:
merged = production.merge(smardlog, left_on="produced_at", right_on="timestamp")

In [12]:
merged["revenue"] = merged["value"] * merged["price"]
tmp1 = merged[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [13]:
tmp1.reset_index(inplace=True)

In [14]:
revenue = tmp1[["plantid", "revenue"]]

In [15]:
plantlist2 = plantlist[["plantid", "energysource"]]
plantlist3 = plantlist2.merge(nat_mp, on="plantid")

In [16]:
tmp0 = pd.merge(revenue, plantlist3, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [17]:
#prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2023].loc[co2s.pollutant == "CO2"]

In [18]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

In [19]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [20]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [21]:
tmp2 = tmp1

In [22]:
#tmp2

In [23]:
coal_cost_per_t = 103.5 or 120
co2_cost = 80
#electricity_price = 78.50

In [24]:
tmp2["co2_cost"] = (tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [25]:
tmp2["profit"] = (tmp2["revenue"] / 10**6) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [26]:
co2s3.dtypes

plantid      object
amount_2    float64
dtype: object

In [27]:
tmp2.sort_values("profit")

,plantid,revenue,energysource,plantname,free_co2s,factor,fuel_price,amount_2,co2_cost,coal_cost,profit
1,BB45025564,7.982573e+08,Braunkohle,Kraftwerk Jänschwalde Block A,11960.0,3.25,18,14.134000,1129.763200,78.280615,-409.786491
26,NW100-0248923,1.093000e+09,Braunkohle,Neurath F,3001.0,3.25,18,16.485000,1318.559920,91.301538,-316.861528
12,BWpf-450-2948214-00000000,1.721634e+08,Steinkohle,GKM Block 6,92027.0,2.68,120,3.430000,267.037840,153.582090,-248.456500
0,BB23020490,1.093048e+08,Mineralölprodukte,1MKA,0.0,2.30,75,3.015000,241.200000,98.315217,-230.210395
34,NW500-0342658,2.653052e+07,Steinkohle,Scholven 1 DT,48161.0,2.68,120,1.993000,155.587120,89.238806,-218.295404
28,NW300-0326774,9.212323e+08,Braunkohle,Niederaußem G,26041.0,3.25,18,13.311000,1062.796720,73.722462,-215.286920
2,BB45025611,6.055226e+08,Braunkohle,Kraftwerk Schwarze Pumpe Block A,207831.0,3.25,18,9.713000,760.413520,53.795077,-208.685953
35,NW500-0915123,1.619746e+08,Steinkohle,Datteln 4,631.0,2.68,120,2.945000,235.549520,131.865672,-205.440545
11,BWpf-450-2797933-00000000,2.106808e+08,Steinkohle,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,2.68,120,3.185000,254.800000,142.611940,-186.731180
14,BYS00041,6.431633e+07,Erdgas,SWM HKW Nord 2 T20,0.0,1.50,40,2.090000,167.200000,55.733333,-158.617001


In [28]:
#tmp2["profit_adj"] = (tmp2["revenue"] * 1.10) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [29]:
tmp2.sort_values("profit")

,plantid,revenue,energysource,plantname,free_co2s,factor,fuel_price,amount_2,co2_cost,coal_cost,profit
1,BB45025564,7.982573e+08,Braunkohle,Kraftwerk Jänschwalde Block A,11960.0,3.25,18,14.134000,1129.763200,78.280615,-409.786491
26,NW100-0248923,1.093000e+09,Braunkohle,Neurath F,3001.0,3.25,18,16.485000,1318.559920,91.301538,-316.861528
12,BWpf-450-2948214-00000000,1.721634e+08,Steinkohle,GKM Block 6,92027.0,2.68,120,3.430000,267.037840,153.582090,-248.456500
0,BB23020490,1.093048e+08,Mineralölprodukte,1MKA,0.0,2.30,75,3.015000,241.200000,98.315217,-230.210395
34,NW500-0342658,2.653052e+07,Steinkohle,Scholven 1 DT,48161.0,2.68,120,1.993000,155.587120,89.238806,-218.295404
28,NW300-0326774,9.212323e+08,Braunkohle,Niederaußem G,26041.0,3.25,18,13.311000,1062.796720,73.722462,-215.286920
2,BB45025611,6.055226e+08,Braunkohle,Kraftwerk Schwarze Pumpe Block A,207831.0,3.25,18,9.713000,760.413520,53.795077,-208.685953
35,NW500-0915123,1.619746e+08,Steinkohle,Datteln 4,631.0,2.68,120,2.945000,235.549520,131.865672,-205.440545
11,BWpf-450-2797933-00000000,2.106808e+08,Steinkohle,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,2.68,120,3.185000,254.800000,142.611940,-186.731180
14,BYS00041,6.431633e+07,Erdgas,SWM HKW Nord 2 T20,0.0,1.50,40,2.090000,167.200000,55.733333,-158.617001
